<a href="https://colab.research.google.com/github/YenliGM/AI-Ticket-Evaluator/blob/main/IA_Engeenier_Asignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Take-Home Assignment: LLM-Based Ticket Reply Evaluation

# LLM-Based Ticket Reply Evaluation

This project provides an automated solution for evaluating AI-generated customer support replies using Google's Gemini LLM. The system assesses replies based on two dimensions: Content (relevance, correctness, completeness) and Format (clarity, structure, grammar).

# Setup Instructions
1. Prerequisites
  * Python 3.10+

  * A Google Gemini API Key (obtainable via Google AI Studio).

2. Dependencies
Install the required libraries using pip:

       Bash
       pip install google-genai pandas


3. Environment Configuration
The script is designed to be flexible with API Key management:

* **Option A**: Enter your key directly in the API_KEY variable inside the script.

* **Option B**: Use Google Colab "Secrets" (icon with a key) and name your secret as GEMINI_API_KEY.

# How to Run
1. Ensure your input file is named tickets.csv and contains the columns ticket and reply.

2. Upload tickets.csv to your Google Colab environment.

3. Run the main cell. The script will process the tickets and automatically run a unit test suite at the end.

4. The results will be generated in tickets_evaluated.csv.

# Evaluation Dimensions
 * Content Score (1-5): Measures factual accuracy and helpfulness.

* Format Score (1-5): Measures grammar, structure, and professional tone.

* Explanations: Provides a brief justification for each score.

# Assignment Code

In [1]:
# ==========================================
# 1. INSTALLATION (Run this once)
# ==========================================
# !pip install -q -U google-genai pandas

import pandas as pd
import time
import os
import csv
import unittest
from google import genai
from google.genai import types

# ==========================================
# 2. SMART API CONFIGURATION
# ==========================================
# OPTION A: Manual Paste (Easy)
# Replace "YOUR_API_KEY_HERE" with your actual key
API_KEY = "YOUR_API_KEY_HERE"

# OPTION B: Google Colab Secrets (Professional)
# If API_KEY is empty or default, it tries to fetch from Colab Secrets
if not API_KEY or "YOUR_API_KEY" in API_KEY:
    try:
        from google.colab import userdata
        API_KEY = userdata.get('GEMINI_API_KEY')
        print("Using API Key from Google Colab Secrets.")
    except Exception:
        print("Manual key not found and Colab Secrets not configured.")

# ==========================================
# 3. CORE LOGIC (AI AGENT MANAGER)
# ==========================================
MODEL_ID = "gemini-2.5-flash"
client = genai.Client(api_key=API_KEY)

def evaluate_ticket_reply(ticket_text, reply_text):
    """
    Evaluates support tickets using Gemini LLM.
    Includes robust error handling and rate limit retries.
    """
    prompt = f"""
    Evaluate the following customer support interaction.

    [Customer Ticket]: {ticket_text}
    [AI Reply]: {reply_text}

    Task: Rate the AI Reply based on:
    1. Content (relevance, correctness, completeness)
    2. Format (clarity, structure, grammar/spelling)

    Output Format (Strictly follow this):
    Content Score: [1-5]
    Content Explanation: [Short text]
    Format Score: [1-5]
    Format Explanation: [Short text]
    """

    for attempt in range(3): # Max 3 retries
        try:
            response = client.models.generate_content(model=MODEL_ID, contents=prompt)
            text_output = response.text

            # Parsing logic
            results = {}
            for line in text_output.split('\n'):
                if ':' in line:
                    key, val = line.split(':', 1)
                    key_clean = key.strip().lower().replace(" ", "_")
                    results[key_clean] = val.strip()

            return {
                'content_score': results.get('content_score', '3'),
                'content_explanation': results.get('content_explanation', 'N/A'),
                'format_score': results.get('format_score', '3'),
                'format_explanation': results.get('format_explanation', 'N/A')
            }
        except Exception as e:
            if "429" in str(e): # Rate limit hit
                wait = 20 * (attempt + 1)
                print(f"Rate limit reached. Waiting {wait}s...")
                time.sleep(wait)
            else:
                return {'content_score': '0', 'content_explanation': f'Error: {e}',
                        'format_score': '0', 'format_explanation': f'Error: {e}'}

def run_main_process():
    input_file = "tickets.csv"
    output_file = "tickets_evaluated.csv"

    if not os.path.exists(input_file):
        print(f"Error: {input_file} not found. Please upload it to Colab.")
        return

    df = pd.read_csv(input_file)
    print(f"Processing {len(df)} tickets...")

    results = []
    for index, row in df.iterrows():
        print(f"Analyzing ticket {index+1}...")
        evaluation = evaluate_ticket_reply(row['ticket'], row['reply'])
        results.append({**row.to_dict(), **evaluation})
        time.sleep(4) # Respect free tier RPM limits

    output_df = pd.DataFrame(results)
    output_df.to_csv(output_file, index=False)
    print(f"Task Complete! Results saved to {output_file}")

# ==========================================
# 4. OPTIONAL: UNIT TESTS
# ==========================================
class TestEvaluation(unittest.TestCase):
    def test_logic_structure(self):
        """Simulate a result to test structure."""
        mock_res = {'content_score': '5', 'format_score': '5'}
        self.assertEqual(mock_res['content_score'], '5')

# ==========================================
# 5. EXECUTION
# ==========================================
if __name__ == "__main__":
    # Run the main process
    run_main_process()

    # Run tests automatically
    print("\n--- Running Automated Tests ---")
    unittest.main(argv=[''], exit=False)

Using API Key from Google Colab Secrets.
Processing 5 tickets...
Analyzing ticket 1...
Analyzing ticket 2...
Analyzing ticket 3...
Analyzing ticket 4...
Analyzing ticket 5...


.
----------------------------------------------------------------------
Ran 1 test in 0.001s

OK


Task Complete! Results saved to tickets_evaluated.csv

--- Running Automated Tests ---
